# Part 3: Add workplace context from Work IQ

Part 3 at Caldova begins with seeded workplace messages about Summit Dose and `CALD-201`. The messages capture current actions and decisions, while the approved sourcing documents define formal commitments. You need to compare them without treating an email or chat as a contractual record.

**🎯 Mission**
- Preserve Caldova's purchasing policy and both **Indexed Knowledge Sources** from Part 1
- Add a **Work IQ Knowledge Source** for recent Summit Dose workplace context
- Compare current actions and decisions with approved `CALD-201` commitments while keeping the evidence types distinct

## The building blocks

In Parts 1 and 2, your knowledge base queried documents and structured data. Now you'll add a source that connects to your personal workplace context.

- **Work IQ Knowledge Source**: connects to Microsoft 365 (Outlook, Teams, Calendar, OneDrive). The KB queries it at runtime to surface emails, chats, and calendar events relevant to the user's question.
- **User assertion**: This notebook uses `InteractiveBrowserCredential` to sign in the current user and request a Work IQ user assertion for `api://<app-id>/access_as_user`. Azure AI Search exchanges that assertion through a federated credential and calls Work IQ on behalf of the signed-in user, honoring Microsoft 365 permissions.

## Step 1: Set up environment variables

Load the configuration for your Foundry IQ (Azure AI Search) and Azure OpenAI resources. The `.env` file in the project folder has the values needed for this part.

In [ ]:
import json
import os

from azure.core.credentials import AzureKeyCredential
from dotenv import load_dotenv

load_dotenv(override=True)


AZURE_SEARCH_SERVICE_ENDPOINT = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
AZURE_SEARCH_ADMIN_KEY = os.environ["AZURE_SEARCH_ADMIN_KEY"]
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"]
AZURE_SEARCH_API_VERSION = "2026-08-01-preview"
AZURE_OPENAI_KEY = os.environ["AZURE_OPENAI_KEY"]
AZURE_OPENAI_CHATGPT_DEPLOYMENT = os.environ["AZURE_OPENAI_CHATGPT_DEPLOYMENT"]
AZURE_OPENAI_CHATGPT_MODEL_NAME = os.environ["AZURE_OPENAI_CHATGPT_MODEL_NAME"]
AZURE_TENANT_ID = os.environ["AZURE_TENANT_ID"]
WORK_IQ_ENTRA_APP_ID = os.environ["WORK_IQ_ENTRA_APP_ID"]
WORK_IQ_FEDERATED_CREDENTIAL_ID = os.environ["WORK_IQ_FEDERATED_CREDENTIAL_ID"]
WORK_IQ_ENTRA_TENANT_ID = os.getenv("WORK_IQ_ENTRA_TENANT_ID", AZURE_TENANT_ID)

SUPPLIER_EVIDENCE_INDEX = "supplier-evidence"
SOURCING_DOCUMENTS_INDEX = "sourcing-documents"

search_credential = AzureKeyCredential(AZURE_SEARCH_ADMIN_KEY)

print("Environment variables loaded")

## Step 2: Login and acquire delegated token

In order for knowledge base retrieval to use your identity with Work IQ, you need an OAuth token for a signed-in user who has access to Microsoft 365 data in this tenant.

Run the cell below to sign in using the credentials in the instructions sidebar. If successful, it acquires a Work IQ user assertion for the `access_as_user` scope on your Work IQ Entra app and saves it for use as `query_work_iq_source_authorization` during retrieval.

In [ ]:
from azure.identity import InteractiveBrowserCredential

user_credential = InteractiveBrowserCredential(
    client_id=WORK_IQ_ENTRA_APP_ID,
    tenant_id=WORK_IQ_ENTRA_TENANT_ID,
)
user_assertion = user_credential.get_token(
    f"api://{WORK_IQ_ENTRA_APP_ID}/access_as_user"
).token

print("Acquired Work IQ user assertion for logged-in user")

## Step 3: Create the indexed and Work IQ knowledge sources

Reuse the direct policy file source from Part 1, recreate the two indexed source definitions, and add Work IQ:

* `file-docs-knowledge-source`: Caldova's uploaded purchasing policy
* `supplier-evidence-knowledge-source`: Supplier invoices, shipment evidence, production yield records, and quality records
* `sourcing-documents-knowledge-source`: Approved RFP responses, agreements, amendments, and purchase orders
* `workiq-knowledge-source`: Recent workplace messages, email, meetings, and files available through Work IQ

In [ ]:
from azure.search.documents.indexes import SearchIndexClient
from azure.search.documents.indexes.models import (
    EntraAppAuthentication,
    SearchIndexFieldReference,
    SearchIndexKnowledgeSource,
    SearchIndexKnowledgeSourceParameters,
    WorkIQKnowledgeSource,
    WorkIQKnowledgeSourceParameters,
)

FILE_KNOWLEDGE_SOURCE_NAME = "file-docs-knowledge-source"
SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME = "supplier-evidence-knowledge-source"
SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME = "sourcing-documents-knowledge-source"
WORK_KNOWLEDGE_SOURCE_NAME = "workiq-knowledge-source"
KNOWLEDGE_SOURCE_NAMES = [
    FILE_KNOWLEDGE_SOURCE_NAME,
    SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME,
    SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
    WORK_KNOWLEDGE_SOURCE_NAME,
]

index_client = SearchIndexClient(endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, credential=search_credential, api_version=AZURE_SEARCH_API_VERSION)

for knowledge_source_name, index_name, description in [
    (
        SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME,
        SUPPLIER_EVIDENCE_INDEX,
        "Caldova supplier invoices, shipment evidence, production yield records, and quality records",
    ),
    (
        SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
        SOURCING_DOCUMENTS_INDEX,
        "Caldova approved RFP responses, agreements, amendments, and purchase orders",
    ),
]:
    knowledge_source = SearchIndexKnowledgeSource(
        name=knowledge_source_name,
        description=description,
        search_index_parameters=SearchIndexKnowledgeSourceParameters(
            search_index_name=index_name,
            source_data_fields=[
                SearchIndexFieldReference(name="blob_path"),
                SearchIndexFieldReference(name="title"),
                SearchIndexFieldReference(name="page_number_from"),
                SearchIndexFieldReference(name="page_number_to"),
            ],
            search_fields=[SearchIndexFieldReference(name="chunk")],
            semantic_configuration_name="semantic-configuration",
        ),
    )
    index_client.create_or_update_knowledge_source(knowledge_source=knowledge_source)

work_knowledge_source = WorkIQKnowledgeSource(
    name=WORK_KNOWLEDGE_SOURCE_NAME,
    description="Caldova workplace context for current Summit Dose actions and decisions",
    work_iq_parameters=WorkIQKnowledgeSourceParameters(
        entra_app_authentication=EntraAppAuthentication(
            application_id=WORK_IQ_ENTRA_APP_ID,
            federated_credential_id=WORK_IQ_FEDERATED_CREDENTIAL_ID,
            tenant_id=WORK_IQ_ENTRA_TENANT_ID,
        )
    ),
)
index_client.create_or_update_knowledge_source(knowledge_source=work_knowledge_source)
print("Knowledge sources created")


## Step 4: Create the multi-source Work IQ knowledge base

This knowledge base preserves the purchasing policy file and both indexed evidence sources, then adds Work IQ for current workplace context.

The knowledge base uses `outputMode=answerSynthesis` and `retrievalReasoningEffort=low`. Its retrieval instructions require approved `CALD-201` commitments to come from sourcing documents and current Summit Dose actions or decisions to come from Work IQ.

In [ ]:
from azure.search.documents.indexes.models import (
    AzureOpenAIVectorizerParameters,
    KnowledgeBase,
    KnowledgeBaseAzureOpenAIModel,
    KnowledgeSourceReference,
)
from azure.search.documents.knowledgebases.models import (
    KnowledgeRetrievalLowReasoningEffort,
    KnowledgeRetrievalOutputMode,
)

KNOWLEDGE_BASE_NAME = "caldova-workiq-knowledge-base"

aoai_params = AzureOpenAIVectorizerParameters(
    resource_url=AZURE_OPENAI_ENDPOINT,
    deployment_name=AZURE_OPENAI_CHATGPT_DEPLOYMENT,
    model_name=AZURE_OPENAI_CHATGPT_MODEL_NAME,
    api_key=AZURE_OPENAI_KEY,
)

knowledge_base = KnowledgeBase(
    name=KNOWLEDGE_BASE_NAME,
    description="Caldova knowledge base combining purchasing policy, indexed procurement evidence, and workplace context",
    models=[KnowledgeBaseAzureOpenAIModel(azure_open_ai_parameters=aoai_params)],
    knowledge_sources=[KnowledgeSourceReference(name=name) for name in KNOWLEDGE_SOURCE_NAMES],
    retrieval_reasoning_effort=KnowledgeRetrievalLowReasoningEffort(),
    output_mode=KnowledgeRetrievalOutputMode.ANSWER_SYNTHESIS,
    retrieval_instructions="Use Work IQ for current Summit Dose actions and decisions, especially equipment mapping, Piedmont subcontractor approval, and PPQ-01 delivery planning. Use approved sourcing documents for CALD-201 commitments. Clearly distinguish workplace discussion from contractual evidence.",
)

index_client.create_or_update_knowledge_base(knowledge_base)
print("Knowledge base created")

## Step 5: Query the knowledge base

Ask Work IQ for current Summit Dose actions and decisions, then compare them with approved `CALD-201` commitments from the sourcing documents.

Workplace messages can describe a technology-transfer milestone, testing-subcontractor evidence, capacity confirmation, or an upcoming supply-review decision. Treat these as current discussion until an approved sourcing document supports them.

> ⏳ Expect to wait 1 to 2 minutes for the response because Work IQ increases retrieval latency. The retrieval request specifies `max_runtime_in_seconds` to increase the time the knowledge base waits for source results.

In [ ]:
from azure.search.documents.knowledgebases import KnowledgeBaseRetrievalClient
from azure.search.documents.knowledgebases.models import (
    FileKnowledgeSourceParams,
    KnowledgeBaseMessage,
    KnowledgeBaseMessageTextContent,
    KnowledgeBaseRetrievalRequest,
    SearchIndexKnowledgeSourceParams,
    WorkIQKnowledgeSourceParams,
)
from IPython.display import Markdown, display

knowledge_base_client = KnowledgeBaseRetrievalClient(
    endpoint=AZURE_SEARCH_SERVICE_ENDPOINT, knowledge_base_name=KNOWLEDGE_BASE_NAME, credential=search_credential, api_version=AZURE_SEARCH_API_VERSION
)

question = (
    "What current Summit Dose updates affect CALD-201? Identify the open equipment-mapping approval, "
    "the missing Piedmont subcontractor evidence and approval decision, and the PPQ-01 April delivery choice. "
    "Compare each with the approved CALD-201 commitments and identify owners, deadlines, and unresolved decisions."
)

retrieval_request = KnowledgeBaseRetrievalRequest(
    messages=[
        KnowledgeBaseMessage(
            role="user",
            content=[KnowledgeBaseMessageTextContent(text=question)],
        )
    ],
    knowledge_source_params=[
        FileKnowledgeSourceParams(
            knowledge_source_name=FILE_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=SUPPLIER_EVIDENCE_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=False,
        ),
        SearchIndexKnowledgeSourceParams(
            knowledge_source_name=SOURCING_DOCUMENTS_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
            always_query_source=True,
        ),
        WorkIQKnowledgeSourceParams(
            knowledge_source_name=WORK_KNOWLEDGE_SOURCE_NAME,
            include_references=True,
            include_reference_source_data=True,
        ),
    ],
    include_activity=True,
    max_runtime_in_seconds=120,
)

result = knowledge_base_client.retrieve(
    retrieval_request=retrieval_request,
    query_work_iq_source_authorization=user_assertion,
)
if not result.response or not result.response[0].content:
    raise RuntimeError("Knowledge base returned no answer content.")
answer_text = getattr(result.response[0].content[0], "text", None)
if not isinstance(answer_text, str):
    raise TypeError("Knowledge base returned an unsupported answer format.")
display(Markdown(answer_text))

### Review the activity log

For this activity log, you will see a "workIQ" step for the call made to Work IQ, along with "searchIndex" steps for each query sent to a search index.

In [ ]:
activities_json = [activity.as_dict() for activity in result.activity or []]
print(json.dumps(activities_json, indent=2))

### Review the references

Work IQ references have `type: "workIQ"`. Their `sourceData` contains `extracts` from workplace context, and references can include `attributions` with links to supporting messages or files. Indexed sourcing references have `type: "searchIndex"` and include document citation metadata.

Use these reference types to keep recent Summit Dose discussion separate from approved `CALD-201` commitments.

In [ ]:
references_json = [ref.as_dict() for ref in result.references or []]
print(json.dumps(references_json, indent=2))

#### 🔍 Source Hunt

Look at the references printed above. Can you identify:

1. Which Work IQ references support the equipment-mapping action, Piedmont approval status, and PPQ-01 delivery decision?
2. Which sourcing documents establish the approved `CALD-201` commitments for transfer, subcontracting, and delivery?
3. Who owns each current action, what is its deadline, and which decisions remain unresolved?
4. Does the answer clearly label workplace discussion separately from approved commitments?

## ✅ Mission complete

**What you built:**

* ✓ **Inherited procurement sources**: Caldova's purchasing policy plus supplier evidence and approved sourcing documents remain available.
* ✓ **Work IQ Knowledge Source**: Recent Summit Dose messages provide current actions, decisions, and unresolved workplace context.
* ✓ **Evidence-aware comparison**: The knowledge base compares current discussion with approved `CALD-201` commitments without treating workplace messages as contractual evidence.

➡️ Continue to: [Part 4: Combine Work IQ and Fabric IQ](part4-work-iq-fabric-iq-to-kb.ipynb).